# Subset health papers at NeurIPS

This notebook downloads all NeurIPS papers on health-related topics

## Setup

In [ ]:
import os

from twentyfiveyears.models import (PublicationVenue,
                                    Publication,
                                    VenueCollection)

In [ ]:
# NeurIPS 2024 health-related papers from papers.nips.cc

import os
import time
from pathlib import Path
from typing import Optional, List

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm  # pip install tqdm if needed

BASE_PROCEEDINGS_URL = "https://papers.nips.cc/paper_files/paper/2024"
BASE_URL = "https://papers.nips.cc"

# Where to save outputs
OUTPUT_DIR = Path("../data/neurips2024_health")
PDF_DIR = OUTPUT_DIR / "pdfs"
OUTPUT_DIR.mkdir(exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

# Optional safety limit while testing (set to None for all papers)
MAX_PAPERS = None  # e.g., 200 during development

# Polite delay between requests (seconds)
REQUEST_SLEEP = 0.2

print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
# Helper function
def http_get(url: str, **kwargs) -> requests.Response:
    """Simple GET wrapper with basic error handling + optional delay."""
    r = requests.get(url, **kwargs)
    if not r.ok:
        raise RuntimeError(f"GET {url} failed [{r.status_code}]: {r.text[:200]}")
    time.sleep(REQUEST_SLEEP)
    return r

## Scrape list of all NeurIPS 2024 papers

In [ ]:
def get_neurips2024_paper_links() -> List[dict]:
    """
    Scrape the NeurIPS 2024 proceedings page and return a list of dicts with:
      - title
      - url (absolute URL to paper page)
    """
    resp = http_get(BASE_PROCEEDINGS_URL)
    soup = BeautifulSoup(resp.text, "html.parser")

    papers = []

    # Heuristic: on papers.nips.cc, each paper appears as an <a> link under the book page.
    # We restrict to links whose href contains "/paper_files/paper/2024/hash/" and "Abstract".
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/paper_files/paper/2024/hash/" in href and "Abstract" in href:
            title = a.get_text(strip=True)
            url = href if href.startswith("http") else BASE_URL + href
            papers.append({"title": title, "url": url})

    # Deduplicate by URL
    seen = {}
    for p in papers:
        seen[p["url"]] = p["title"]
    papers = [{"url": url, "title": title} for url, title in seen.items()]

    print(f"Found {len(papers)} paper links for NeurIPS 2024.")
    return sorted(papers, key=lambda x: x["title"].lower())


paper_links = get_neurips2024_paper_links()

if MAX_PAPERS is not None:
    paper_links = paper_links[:MAX_PAPERS]
    print(f"Limiting to first {len(paper_links)} papers for testing.")

paper_links[:5]


## Parse an individual paper page (title, authors, abstract, DOI, PDF)

In [ ]:
def extract_text_after_heading(soup: BeautifulSoup, heading_text: str) -> Optional[str]:
    """
    Given a BeautifulSoup document and a heading string like 'Abstract' or 'Authors',
    find the heading (h4) and return the text of the next sibling element.
    """
    h = soup.find(["h3", "h4"], string=lambda s: s and s.strip().lower() == heading_text.lower())
    if not h:
        return None

    # Most pages use a <p> right after the heading
    sib = h.find_next_sibling()
    if sib is None:
        return None
    return sib.get_text(" ", strip=True)


def parse_paper_page(url: str) -> dict:
    """
    Parse a NeurIPS 2024 paper page and extract:
      - title
      - authors (string)
      - abstract
      - doi
      - pdf_url
    """
    resp = http_get(url)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Title: the page has the title as an h4 just under the top
    title_el = soup.find(["h2", "h3", "h4"])
    page_title = title_el.get_text(strip=True) if title_el else None

    # Authors
    authors = extract_text_after_heading(soup, "Authors")

    # DOI: there is a "Digital Object Identifier (DOI)" section with a link
    doi = None
    doi_heading = soup.find(["h3", "h4"], string=lambda s: s and "digital object identifier" in s.lower())
    if doi_heading:
        doi_link = doi_heading.find_next("a", href=True)
        if doi_link:
            doi = doi_link.get_text(strip=True)

    # Abstract
    abstract = extract_text_after_heading(soup, "Abstract")

    # PDF link: there is a link labeled "Paper"
    pdf_url = None
    link = soup.find("a", string=lambda s: s and s.strip().lower() == "paper")
    if link and link.has_attr("href"):
        href = link["href"]
        pdf_url = href if href.startswith("http") else BASE_URL + href

    return {
        "page_url": url,
        "page_title": page_title,
        "authors": authors,
        "doi": doi,
        "abstract": abstract,
        "pdf_url": pdf_url,
    }


# Quick smoke test on the first paper
example = parse_paper_page(paper_links[0]["url"])
example


## Health classification rules (title + abstract)

In [ ]:
HEALTH_TEXT_KEYWORDS = [
    "health", "healthcare", "health care", "medical", "medicine", "clinical",
    "hospital", "patient", "patients", "electronic health record", "ehr",
    "intensive care", "icu", "ward", "radiology", "ct", "mri", "x-ray", "xray",
    "biomedical", "bio-medical", "bioinformatics", "genomic", "genomics",
    "proteomic", "protein", "drug", "pharmacology", "therapeutic", "therapy",
    "disease", "diagnosis", "screening", "epidemiology", "public health",
    "covid", "sepsis", "mortality", "clinical trial", "trial",
    "ecg", "electrocardiogram", "eeg", "electroencephalogram",
]

def is_health_related_title_abstract(title: Optional[str], abstract: Optional[str]) -> bool:
    text = f"{title or ''} {abstract or ''}".lower()
    return any(kw in text for kw in HEALTH_TEXT_KEYWORDS)


## Crawl all paper pages and build a table

In [ ]:
records = []

for p in tqdm(paper_links, desc="Fetching and parsing paper pages"):
    url = p["url"]
    meta = parse_paper_page(url)

    title = meta["page_title"] or p["title"]
    abstract = meta["abstract"]

    health_flag = is_health_related_title_abstract(title, abstract)

    records.append({
        "title": title,
        "authors": meta["authors"],
        "doi": meta["doi"],
        "abstract": abstract,
        "page_url": meta["page_url"],
        "pdf_url": meta["pdf_url"],
        "is_health": health_flag,
    })

df = pd.DataFrame(records)
print("Total papers parsed:", len(df))
print("Health-related papers:", df["is_health"].sum())
df.head()


## Save all papers and health subset

In [ ]:
all_csv = OUTPUT_DIR / "neurips2024_all_from_neurips_site.csv"
health_csv = OUTPUT_DIR / "neurips2024_health_from_neurips_site.csv"

df.to_csv(all_csv, index=False)

health_df = df[df["is_health"]].copy()
health_df.to_csv(health_csv, index=False)

print("Saved:")
print(" - All papers   :", all_csv)
print(" - Health papers:", health_csv)
print("Health subset size:", len(health_df))

health_df[["title", "authors", "doi"]].head(20)


## Download PDFs for health-related papers

In [ ]:
HEALTH_TEXT_KEYWORDS = [
    "health", "healthcare", "health care", "medical", "medicine", "clinical",
    "hospital", "patient", "patients", "electronic health record", "ehr",
    "intensive care", "icu", "ward", "radiology", "ct", "mri", "x-ray", "xray",
    "biomedical", "bio-medical", "bioinformatics", "genomic", "genomics",
    "proteomic", "protein", "drug", "pharmacology", "therapeutic", "therapy",
    "disease", "diagnosis", "screening", "epidemiology", "public health",
    "covid", "sepsis", "mortality", "clinical trial", "trial",
    "ecg", "electrocardiogram", "eeg", "electroencephalogram",
]

def is_health_related_title_abstract(title: Optional[str], abstract: Optional[str]) -> bool:
    text = f"{title or ''} {abstract or ''}".lower()
    return any(kw in text for kw in HEALTH_TEXT_KEYWORDS)
